# Project 2: Customer Segmentation & RFM Analysis
## Austin Airbnb Host Portfolio

**Analyst framing.** Airbnb operates as a two-sided marketplace. While guests generate bookings, hosts supply the inventory that makes the platform viable. This analysis treats **hosts as Airbnb's customers** — applying RFM segmentation and K-Means clustering to the Austin host base to identify retention risks, revenue concentrations, and growth opportunities.

**Business questions addressed**
- Which hosts constitute the top revenue tier, and how concentrated is marketplace value?
- Which previously-active hosts show signs of disengagement and warrant retention outreach?
- Which host segments represent underdeveloped growth potential?

**Methodology**
1. Data loading and RFM-relevant column audit
2. Host-level aggregation and raw RFM calculation
3. Quintile scoring
4. Rule-based segmentation
5. K-Means clustering with silhouette-based k selection
6. Segment profiling
7. Business recommendations

**Dataset.** `data/listings_clean.csv` — 10,402 Austin listings, cleaned via Project 1 pipeline. Source: [Inside Airbnb](http://insideairbnb.com/austin).

## 1. Data Loading & Initial Inspection

Objective: verify dataset structure, confirm presence and quality of columns required for RFM construction, and surface any data-quality issues that will affect methodology decisions in Section 2.

In [1]:
# Core data stack
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utility
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display configuration
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Visualization defaults — IBCS-inspired minimal styling
sns.set_style("white")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(f"Pandas version:       {pd.__version__}")
print(f"NumPy version:        {np.__version__}")
print(f"Notebook initialized: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Pandas version:       2.2.3
NumPy version:        2.2.4
Notebook initialized: 2026-04-19 13:25


In [2]:
# Load the cleaned listings dataset from Project 1
df = pd.read_csv('data/listings_clean.csv')

print(f"Shape:  {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head(3)

Shape:  10,402 rows × 54 columns
Memory: 7.93 MB


,id,name,host_id,host_name,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,availability_30,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_cleanliness,review_scores_location,instant_bookable,calculated_host_listings_count,reviews_per_month,host_experience_years,has_wifi,has_kitchen,has_free_parking,has_paid_parking,has_ac,has_washer,has_outdoor_dining,has_pets_allowed,has_bbq,has_fire_pit,has_private_entrance,has_pool,has_gym,has_ev_charger,has_hot_tub,amenity_count,price_per_bedroom,is_new_listing
0,5456,"Walk to 6th, Rainey St and Convention Ctr",8028,Sylvia,2009-02-16,within a few hours,100.00,90.00,True,True,78702,30.26,-97.73,Entire guesthouse,Entire home/apt,3,1.00,1.00,2.00,97.00,2,13,328,708,25,150,"14,550.00",2009-03-19,2025-09-02,4.85,4.86,4.73,f,1,3.52,17.20,True,True,False,False,True,False,False,False,False,False,True,False,False,False,False,26,97.00,False
1,6448,"Secluded Studio @ Zilker - King Bed, Bright & ...",14156,Amy,2009-04-20,within an hour,100.00,96.00,True,True,78704,30.26,-97.76,Entire guesthouse,Entire home/apt,2,1.00,1.00,2.00,160.00,3,12,316,339,14,84,"13,440.00",2011-09-06,2025-08-20,4.97,4.96,4.97,t,1,1.98,17.00,True,True,True,False,True,True,True,False,False,False,True,False,False,False,False,61,160.00,False
2,8502,Woodland Studio Lodging,25298,Karen,2009-07-11,within a day,100.00,60.00,False,False,78741,30.23,-97.74,Entire guest suite,Entire home/apt,2,1.00,1.00,1.00,38.00,4,29,88,54,1,8,304.00,2010-02-19,2025-05-05,4.57,4.67,4.69,f,1,0.28,16.80,True,True,False,False,True,False,False,True,False,False,False,False,False,False,False,12,38.00,False


In [3]:
# Columns required for RFM construction, plus contextual fields
rfm_columns = [
    'host_id',                          # Grouping key — one host = one customer
    'host_name',                        # Interpretability for segment profiles
    'last_review',                      # Recency source
    'first_review',                     # Host tenure context
    'number_of_reviews',                # Frequency: lifetime reviews
    'number_of_reviews_ltm',            # Frequency: trailing 12 months
    'reviews_per_month',                # Frequency: activity rate
    'price',                            # Contextual pricing
    'estimated_revenue_l365d',          # Monetary: pre-computed annual revenue proxy
    'estimated_occupancy_l365d',        # Monetary: bookable-night volume
    'calculated_host_listings_count',   # Portfolio size
    'availability_365',                 # Capacity context
    'is_new_listing'                    # Flag: listings with no review history
]

# Verify presence
missing = [c for c in rfm_columns if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

# Column-level audit
audit = pd.DataFrame({
    'dtype': df[rfm_columns].dtypes.astype(str),
    'non_null': df[rfm_columns].notna().sum(),
    'nulls': df[rfm_columns].isna().sum(),
    'null_pct': (df[rfm_columns].isna().sum() / len(df) * 100).round(2)
})
audit

,dtype,non_null,nulls,null_pct
host_id,int64,10402,0,0.00
host_name,object,10402,0,0.00
last_review,object,10402,0,0.00
first_review,object,10402,0,0.00
number_of_reviews,int64,10402,0,0.00
number_of_reviews_ltm,int64,10402,0,0.00
reviews_per_month,float64,10402,0,0.00
price,float64,10402,0,0.00
estimated_revenue_l365d,float64,10402,0,0.00
estimated_occupancy_l365d,int64,10402,0,0.00


**Column availability and data integrity.** All thirteen RFM-relevant columns are present in the dataset, with zero null values recorded across 10,402 observations. No imputation, dropping, or flagging of missing values is required before proceeding to the aggregation stage. This level of completeness is unusual for raw marketplace data and reflects the cleaning work completed in Project 1.

**Data type observations.** Numeric fields (`number_of_reviews`, `price`, `estimated_revenue_l365d`, etc.) are correctly typed as integers or floats and require no conversion. The boolean flag `is_new_listing` is properly typed. However, the two date fields — `last_review` and `first_review` — are stored as `object` (string) type. Datetime conversion will be required before Recency can be calculated in days. This conversion will be handled in Section 2.

**Forward-looking consideration.** While null counts are clean, the `last_review` field contains a placeholder value (`"No reviews yet"`) for listings with no review history. This placeholder does not register as a null but behaves as one functionally, and will require explicit handling during datetime conversion.

In [4]:
# Host-level concentration
n_listings = len(df)
n_hosts = df['host_id'].nunique()
avg_listings_per_host = n_listings / n_hosts

print("Host-level concentration")
print("-" * 40)
print(f"Total listings:      {n_listings:,}")
print(f"Unique hosts:        {n_hosts:,}")
print(f"Avg listings/host:   {avg_listings_per_host:.2f}")

# Date range of review activity (excluding placeholder "No reviews yet")
valid_reviews = df.loc[df['last_review'] != 'No reviews yet', 'last_review']
lr = pd.to_datetime(valid_reviews, errors='coerce')

print("\nReview date range")
print("-" * 40)
print(f"Earliest last_review:  {lr.min().date()}")
print(f"Latest last_review:    {lr.max().date()}")
print(f"Listings with reviews: {len(valid_reviews):,}")
print(f"Listings without:      {(df['last_review'] == 'No reviews yet').sum():,}")

Host-level concentration
----------------------------------------
Total listings:      10,402
Unique hosts:        5,503
Avg listings/host:   1.89

Review date range
----------------------------------------
Earliest last_review:  2011-03-21
Latest last_review:    2025-09-16
Listings with reviews: 8,850
Listings without:      1,552


**Host concentration.** The dataset contains 10,402 listings distributed across 5,503 unique hosts, yielding an average of 1.89 listings per host. This confirms that a substantial share of listings belong to multi-property hosts, validating the decision to aggregate analysis to the host level rather than analyzing listings independently. From a marketplace perspective, losing a single multi-property host represents a materially larger retention risk than losing an individual listing — an asymmetry that listing-level analysis would obscure.

**Review activity window.** The earliest recorded review in the dataset occurred on 2011-03-21 and the most recent on 2025-09-16. This fourteen-year window captures Airbnb's full Austin market lifecycle, including pre-pandemic growth, the 2020 disruption, and the subsequent recovery. The latest date, **2025-09-16, establishes the reference point for Recency calculation** — all host recency scores will be measured as days elapsed from this snapshot date to each host's most recent review.

**Coverage of review-based metrics.** Of the 10,402 listings, 8,850 (85.1%) have review history and can contribute directly to Recency and Frequency calculations. The remaining 1,552 listings (14.9%) have never been reviewed and are flagged as new listings. The treatment of these listings within the RFM framework is the central methodology decision to be formalized in Section 2, with three candidate approaches: exclusion, inclusion with a maximum-recency penalty, or segmentation as a distinct new-host cohort.